# Host the fine-tuned MARS INTELLIGENCE Qwen on Colab

Runtime > Change runtime type > **T4 GPU**, then Runtime > **Run all**. Cell 3 prints `QWEN_BASE_URL` and
`QWEN_API_KEY`; give both to the NeuroNexus backend (see `tools/codespace_qwen_demo.sh`).

- Serves `Mhaquehaque/neuronexus-qwen3-4b-mars-intel` (Q4_K_M) with llama.cpp on the GPU, behind a free
  Cloudflare quick tunnel (no account), protected by a random API key.
- Temporary: the URL changes every run and stops when the session ends. Colab's terms restrict serving web
  traffic, so use it for testing or a demo window, not as always-on hosting.

In [ ]:
%%bash
# 1. Build llama.cpp with CUDA for this GPU, download the model and cloudflared (about 8 to 12 min once)
set -e
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
cd /content
[ -d llama.cpp ] || git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
if [ ! -x llama.cpp/build/bin/llama-server ]; then
  cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=native -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release > build.log 2>&1
  cmake --build llama.cpp/build -j "$(nproc)" --target llama-server >> build.log 2>&1 || { tail -30 build.log; exit 1; }
fi
echo "llama-server built"
[ -s qwen4b.gguf ] || curl -sL --fail -o qwen4b.gguf https://huggingface.co/Mhaquehaque/neuronexus-qwen3-4b-mars-intel/resolve/main/neuronexus-qwen3-4b-mars-intel-Q4_K_M.gguf
ls -lh qwen4b.gguf
[ -x cloudflared ] || { curl -sL --fail -o cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared; }
./cloudflared --version

In [ ]:
# 2. Start the model server on the GPU, with an API key
import re, secrets, subprocess, time, requests
QWEN_API_KEY = secrets.token_urlsafe(24)
server = subprocess.Popen(
    ["/content/llama.cpp/build/bin/llama-server", "-m", "/content/qwen4b.gguf", "--jinja", "-c", "4096", "-np", "1",
     "-ngl", "99", "--host", "127.0.0.1", "--port", "8081", "--api-key", QWEN_API_KEY,
     "--alias", "neuronexus-qwen3-4b-mars-intel"],
    stdout=open("/content/llama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        if requests.get("http://127.0.0.1:8081/health", timeout=2).status_code == 200:
            break
    except requests.RequestException:
        pass
    time.sleep(1)
print("llama-server:", requests.get("http://127.0.0.1:8081/health", timeout=5).json())

In [ ]:
# 3. Open a free public HTTPS tunnel and print what the backend needs
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8081"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
QWEN_BASE_URL = None
deadline = time.time() + 90
while time.time() < deadline and not QWEN_BASE_URL:
    line = tunnel.stdout.readline()
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        QWEN_BASE_URL = m.group(0)
assert QWEN_BASE_URL, "No tunnel URL; rerun this cell"
for _ in range(30):  # DNS for a new quick tunnel takes a few seconds
    try:
        if requests.get(f"{QWEN_BASE_URL}/health", timeout=5).status_code == 200:
            break
    except requests.RequestException:
        pass
    time.sleep(2)
print(f"export QWEN_BASE_URL={QWEN_BASE_URL}")
print(f"export QWEN_API_KEY={QWEN_API_KEY}")

In [ ]:
# 4. Live test through the public URL (what the backend will do)
ctx = ("# terrain analysis\nsite terrain elevation m: -282 [MOLA_MEGDR]\nsite terrain slope deg: 28.87 [MOLA_MEGDR]\n"
       "note: terrain ladder [NEURONEXUS_TERRAIN_LADDER]: HiRISE DTM 1 m, CTX DTM 20 m, MOLA 128 ppd about 463 m per pixel")
system = requests.get("https://raw.githubusercontent.com/kmmohaimenulhaque-code/martian-map/"
                      "feature/mission-console-v2-Qwen-Gem/training/data/system_prompt.txt", timeout=30).text.strip()
t = time.time()
r = requests.post(f"{QWEN_BASE_URL}/v1/chat/completions", timeout=120,
                  headers={"Authorization": f"Bearer {QWEN_API_KEY}"},
                  json={"model": "neuronexus-qwen3-4b-mars-intel", "temperature": 0.2, "max_tokens": 200,
                        "chat_template_kwargs": {"enable_thinking": False},
                        "messages": [{"role": "system", "content": system},
                                     {"role": "user", "content": f"CONTEXT:\n{ctx}\n\nQUESTION: How steep is it here?"}]})
r.raise_for_status()
print(f"{time.time() - t:.1f} s:", r.json()["choices"][0]["message"]["content"])

In [ ]:
# 5. Keep this cell running to hold the session (stop it, or close the tab, to shut the host down)
while server.poll() is None and tunnel.poll() is None:
    time.sleep(60)
print("server or tunnel stopped; rerun cells 2 and 3")